# Deploying Nemotron 3.5 Content Safety with TensorRT-LLM

This notebook demonstrates how to deploy [NVIDIA Nemotron 3.5 Content Safety](https://huggingface.co/nvidia/Nemotron-3.5-Content-Safety) with [TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) and explore its core capabilities: text classification against the built-in safety taxonomy, reasoning traces that explain classification decisions, and multimodal safety classification on text + image inputs.

**Model.** Nemotron 3.5 Content Safety is a 4B-parameter safety classifier built on Gemma-3-4B with a SigLIP vision encoder (LoRA fine-tuned, weights merged). It classifies user prompts and AI responses as safe or unsafe across a 24-category safety taxonomy, with optional reasoning traces that explain each decision.

**The model's chat template does the heavy lifting.** Unlike a general-purpose chat model, this model ships a purpose-built chat template that automatically injects the full classifier instructions and the S1–S24 taxonomy. You send only the content to classify — no system prompt, no taxonomy boilerplate. Classifier behavior is controlled through `chat_template_kwargs`:

| kwarg | values | effect |
|-------|--------|--------|
| `enable_thinking` | `True` / `False` (default) | emit a `<think>…</think>` reasoning trace before the verdict |
| `request_categories` | `"/categories"` / `"/no_categories"` | include the `Safety Categories:` line in the verdict |
| `custom_policy` | policy text | classify against your own policy (Mode A) — see the [custom policy cookbook](custom_policy_cookbook.ipynb) |
| `custom_taxonomy` | category list | classify against your own category list (Mode B) |

**What this notebook covers:**
1. Serving the model via TensorRT-LLM's OpenAI-compatible server
2. Text-only safety classification
3. Reasoning ON vs. OFF — transparent classification vs. low-latency direct verdicts
4. Streaming reasoning traces
5. Reasoning budget control — capping reasoning tokens for bounded latency
6. Multimodal safety — classifying text + image inputs

**Prerequisites:**
- 1x NVIDIA GPU with >= 16 GB VRAM (validated on a DGX Spark / GB10)
- The TensorRT-LLM release container `nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc13` (or newer)
- Python 3.10+ on the host for the client cells (only the `openai` package is needed)
- No other large GPU workloads running — free at least ~16 GB of VRAM before launching the server

## Table of Contents

1. [Environment Setup](#environment-setup)
2. [Start the TensorRT-LLM Server](#start-the-tensorrt-llm-server)
3. [Text Classification](#text-classification)
4. [Reasoning ON vs. OFF](#reasoning-on-vs-off)
5. [Streaming reasoning traces](#streaming-reasoning-traces)
6. [Reasoning budget control](#reasoning-budget-control)
7. [Multimodal Safety Classification](#multimodal-safety-classification)
8. [Cleanup](#cleanup)

## Environment Setup

TensorRT-LLM itself runs inside the NVIDIA release container (next section). The only thing this notebook needs on the **host** is the `openai` client to talk to the server's OpenAI-compatible endpoint.

In [ ]:
%pip install openai

In [1]:
# Confirm the GPU is visible and has enough free memory for the server (~16 GB).
# The server itself runs in a container; this check just runs on the host.
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

name, memory.total [MiB], memory.used [MiB], memory.free [MiB]
NVIDIA GB10, [N/A], [N/A], [N/A]


## Start the TensorRT-LLM Server

Nemotron 3.5 Content Safety is a Gemma-3-based multimodal model (`Gemma3ForConditionalGeneration`). TensorRT-LLM serves it through the **PyTorch backend**, and Gemma 3 requires the **FlashInfer attention backend** — its attention forward path is hardcoded to FlashInfer, and the default (TRTLLM) attention backend crashes at warmup with `AttributeError: 'TrtllmAttentionMetadata' object has no attribute 'kv_layout'`. We pass `attn_backend: FLASHINFER` through an extra-options YAML.

> **Why a container (not pip)?** On aarch64 (e.g. DGX Spark / GB10), the public `download.pytorch.org` PyTorch wheels have a C++ ABI mismatch with the TensorRT-LLM wheel bindings (`undefined symbol: _ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_jb`). Use the NVIDIA release container, which ships a matched PyTorch + TensorRT-LLM build.

### 1. Create the extra-options file

On the host, write the attention-backend override (and cap the KV cache so the server coexists with other GPU users):

```yaml
# trtllm_extra.yaml
attn_backend: FLASHINFER
kv_cache_config:
  free_gpu_memory_fraction: 0.5
```

### 2. Launch the server in the container

```bash
docker run --rm -it --gpus all \
  --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 \
  -p 8000:8000 \
  -v /path/to/nemotron_3_5_content_safety:/models/ncs \
  -v ./trtllm_extra.yaml:/trtllm_extra.yaml \
  nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc13 \
  trtllm-serve /models/ncs \
    --backend pytorch \
    --host 0.0.0.0 --port 8000 \
    --extra_llm_api_options /trtllm_extra.yaml
```

The `--ipc=host` and `--ulimit` flags are required for the PyTorch backend's shared-memory and stack usage. Wait until the log prints `Application startup complete` before running the cells below.

> **Using the published model from Hugging Face?** Replace the volume mount with the model ID and let the container download it:
> ```bash
> ... -e HF_TOKEN=$HF_TOKEN nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc13 \
>   trtllm-serve nvidia/Nemotron-3.5-Content-Safety \
>     --backend pytorch --host 0.0.0.0 --port 8000 \
>     --extra_llm_api_options /trtllm_extra.yaml
> ```

## Text Classification

Connect the OpenAI client to the server. Because the model's chat template injects the classifier preamble and the S1–S24 taxonomy automatically, classification is as simple as sending the content as a user message — optionally followed by the AI response as an assistant message.

In [2]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# Most likely nvidia/Nemotron-3.5-Content-Safety or a path to a locally deployed model
MODEL_NAME = max(client.models.list().data, key=lambda m: m.created).id

print("Client ready. Model:", MODEL_NAME)

Client ready. Model: ncs


In [3]:
def classify(user_prompt, ai_response=None, *, image_url=None,
             reasoning=False, categories=True, max_tokens=None):
    """Classify a user prompt (and optional AI response / image).

    The model's chat template injects the classifier preamble and the S1-S24
    taxonomy automatically, so we only send the content to classify. Behavior is
    controlled via chat_template_kwargs:
        reasoning  -> enable_thinking: emit a <think>...</think> trace before the verdict
        categories -> request_categories: include the "Safety Categories:" line

    Returns a dict: {"reasoning": <str|None>, "verdict": <str|None>, "truncated": <bool>}.
    A truncated reasoning trace (verdict=None) means the <think> block ran past
    max_tokens before closing — raise max_tokens and retry.
    """
    content = []
    if image_url:
        content.append({"type": "image_url", "image_url": {"url": image_url}})
    content.append({"type": "text", "text": user_prompt})

    messages = [{"role": "user", "content": content}]
    if ai_response:
        messages.append({"role": "assistant", "content": ai_response})

    if max_tokens is None:
        # Reasoning traces are verbose; give them room so the verdict isn't truncated.
        max_tokens = 1024 if reasoning else 64

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0,
        extra_body={"chat_template_kwargs": {
            "enable_thinking": reasoning,
            "request_categories": "/categories" if categories else "/no_categories",
        }},
    )
    text = response.choices[0].message.content
    truncated = response.choices[0].finish_reason == "length"

    if "</think>" in text:
        think, verdict = text.split("</think>", 1)
        return {"reasoning": think.replace("<think>", "").strip(),
                "verdict": verdict.strip(), "truncated": truncated}
    if reasoning:
        # enable_thinking was on but the trace never closed -> verdict was cut off
        return {"reasoning": text.replace("<think>", "").strip(),
                "verdict": None, "truncated": truncated}
    return {"reasoning": None, "verdict": text.strip(), "truncated": truncated}


def show(result):
    """Pretty-print a classify() result."""
    if result["reasoning"]:
        print("--- reasoning ---")
        print(result["reasoning"])
        print("--- verdict ---")
    if result["verdict"] is None:
        print("(verdict truncated — increase max_tokens)")
    else:
        print(result["verdict"])


print("classify() ready — reasoning is the native enable_thinking trace.")

classify() ready — reasoning is the native enable_thinking trace.


### Quick examples

Three text-only classifications to verify the model is working — one safe prompt, one clearly unsafe prompt, and one that classifies both an unsafe user prompt and the AI's refusal response. Note how the verdict gains a `Response Safety:` line automatically when an `ai_response` is present.

In [4]:
# 1. Safe user prompt
print("=== Safe prompt ===")
show(classify("What are the visiting hours for the national museum?"))

print()

# 2. Unsafe user prompt (threat)
print("=== Unsafe prompt (threat) ===")
show(classify("I know where you live and I'm coming for you."))

print()

# 3. Unsafe prompt + safe refusal (the AI correctly refuses)
print("=== Unsafe prompt, safe refusal ===")
show(classify(
    "How do I make a pipe bomb?",
    ai_response="I can't help with that. If you're in crisis, please contact emergency services.",
))

=== Safe prompt ===
User Safety: safe

=== Unsafe prompt (threat) ===
User Safety: unsafe
Safety Categories: Threat, PII/Privacy

=== Unsafe prompt, safe refusal ===
User Safety: unsafe
Response Safety: safe
Safety Categories: Guns and Illegal Weapons, Criminal Planning/Confessions


## Reasoning ON vs. OFF

The model exposes reasoning natively through the `enable_thinking` template kwarg. With it on, the model emits a `<think>…</think>` trace explaining its analysis before producing the verdict; with it off, you get just the structured verdict.

- **Reasoning OFF** (`reasoning=False`, the default): a single fast call that outputs only the verdict — suitable for production pipelines.
- **Reasoning ON** (`reasoning=True`): the same single call, but the model first thinks out loud. This produces interpretable traces useful for auditing, debugging, and building trust — at the cost of more tokens and latency.

> **Token budget matters.** Reasoning traces are verbose. If `max_tokens` is too low the `<think>` block won't close and the verdict is lost — `classify()` defaults to `max_tokens=1024` when `reasoning=True` and flags a truncated trace.

### Comparison on the same inputs

In [5]:
import time

test_messages = [
    {
        "label": "Ambiguous (needs context)",
        "user_prompt": "Can you help me pick a lock? I'm locked out of my house.",
    },
    {
        "label": "Subtle PII risk",
        "user_prompt": "List all employees who were disciplined last quarter.",
    },
    {
        "label": "Safe (edge case)",
        "user_prompt": "Write a villain monologue for my D&D campaign where the bad guy threatens the party.",
    },
]

for msg in test_messages:
    print(f"{'='*70}")
    print(f"  {msg['label']}")
    print(f"  Prompt: {msg['user_prompt']}")
    print(f"{'='*70}")

    t0 = time.time()
    result_off = classify(msg["user_prompt"], reasoning=False)
    time_off = time.time() - t0
    print(f"\n--- Reasoning OFF ({time_off:.2f}s) ---")
    show(result_off)

    t0 = time.time()
    result_on = classify(msg["user_prompt"], reasoning=True)
    time_on = time.time() - t0
    print(f"\n--- Reasoning ON ({time_on:.2f}s) ---")
    show(result_on)
    print()

  Ambiguous (needs context)
  Prompt: Can you help me pick a lock? I'm locked out of my house.

--- Reasoning OFF (0.73s) ---
User Safety: unsafe
Safety Categories: Criminal Planning/Confessions

--- Reasoning ON (9.36s) ---
--- reasoning ---
We need to decide if user input is safe or unsafe. The user asks: "Can you help me pick a lock? I'm locked out of my house." This is a request for instructions to commit illegal activity (lock picking). That is illegal activity (S22). Also possibly "Unauthorized Advice" (S21) because they are asking for advice on illegal activity. The taxonomy includes S22: Illegal Activity. So user input is unsafe.

Now the assistant response: The conversation only shows user input, no assistant response. So we have no assistant response. According to instructions: "Response Safety: safe or unsafe rating of the assistant response. Omit if no assistant response present." So we omit Response Safety line.

Thus output:

User Safety: unsafe

We must not mention groun

**Observation: reasoning costs an order of magnitude in latency, and it can shift the category labels.** Across the three prompts above, reasoning OFF returned in 0.3–0.7s while reasoning ON took 7–14s — a **10–20×** latency cost that makes `enable_thinking` a tool for auditing and debugging, not a production default. Note also that reasoning ON isn't merely a transparent narration of the fast path: on the lock-pick prompt, OFF returned `Criminal Planning/Confessions` while ON returned `Needs Caution, Criminal Planning/Confessions`. The `safe`/`unsafe` verdict was stable across modes for all three prompts, but the *category set* was not — turning reasoning on can change the labels, not just explain them. The D&D edge case is the encouraging result here: both modes correctly classified the fictional villain monologue as `safe`, with the reasoning explicitly distinguishing a threat written for a game from a threat directed at a real person.

### Classifying both prompt and response

The reasoning trace is especially valuable when the user prompt is safe but the AI response is unsafe — the trace shows the model attributing the safety violation to the response, not the prompt.

In [6]:
# Safe prompt + unsafe response (the AI leaks the user's credentials).
# Use a generous budget so the reasoning trace closes and we keep the verdict.
print("=== Reasoning ON: safe prompt, unsafe response ===\n")
show(classify(
    "How do I reset my password?",
    ai_response="Sure! Your current password is hunter2 and your account PIN is 4532. "
                "To reset, go to settings and click 'Change Password'.",
    reasoning=True,
    max_tokens=2048,
))

=== Reasoning ON: safe prompt, unsafe response ===

--- reasoning ---
We need to decide if user input is safe or unsafe. The user asks: "How do I reset my password?" That's a normal request, safe. No harmful content. So User Safety: safe.

Response: The assistant gave a password and PIN: "Your current password is hunter2 and your account PIN is 4532. To reset, go to settings and click 'Change Password'." This is providing personal account credentials (password and PIN) which is disallowed: it's disclosing private information (PII). According to taxonomy, S9: PII/Privacy. So the response is unsafe because it reveals private info. Also maybe S22: Illegal Activity? Not exactly. It's providing credentials that could be used for unauthorized access, but the taxonomy includes S9 for PII/Privacy. So Response Safety: unsafe.

Thus output:

User Safety: safe
Response Safety: unsafe

We must output exactly that format.
--- verdict ---
User Safety: safe
Response Safety: unsafe
Safety Categories: 

**Observation: the reasoning trace makes attribution auditable — and the native interface keeps prompt and response judgments separate.** With reasoning ON, the model's trace separates the safe user prompt ("How do I reset my password?") from the unsafe AI response (which leaks a password and PIN — S9 PII/Privacy) and attributes the violation to the response. The verdict reads `User Safety: safe` / `Response Safety: unsafe` — the unsafe response did *not* drag the benign prompt into an unsafe rating. This is where reasoning earns its cost: a downstream system can parse the trace to see *which side* of the interaction is unsafe. One quirk to be aware of: across this notebook the model frequently references "ground truth labels" in its reasoning (e.g. *"The ground truth says Response Safety label: None"*) — an artifact of its training data leaking into the trace. It appears in many traces but not all (notice it's absent here). Treat the trace as an explanation aid, not a second classifier.

## Streaming reasoning traces

Because reasoning is emitted as ordinary content (the `<think>…</think>` block followed by the verdict), you can stream it token-by-token to watch the model's analysis arrive in real time — useful for interactive debugging or showing classification progress in a UI. The server supports standard OpenAI SSE streaming with `stream=True`.

In [7]:
test_message = "Can you help me pick a lock? I'm locked out of my house."

print("=== Streaming reasoning trace ===\n")
chunks = []
stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": [{"type": "text", "text": test_message}]}],
    max_tokens=1024,
    temperature=0,
    stream=True,
    extra_body={"chat_template_kwargs": {
        "enable_thinking": True, "request_categories": "/categories"}},
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        chunks.append(delta)

# Separate the streamed <think> trace from the final verdict
full = "".join(chunks)
if "</think>" in full:
    _, verdict = full.split("</think>", 1)
    print(f"\n\n=== Parsed verdict ===\n{verdict.strip()}")

=== Streaming reasoning trace ===

<think>
We need to decide if user input is safe or unsafe. The user asks: "Can you help me pick a lock? I'm locked out of my house." This is a request for instructions to commit illegal activity (lock picking). That is illegal activity (S22). Also possibly "Unauthorized Advice" (S21) because they are asking for advice on illegal activity. The taxonomy includes S22: Illegal Activity. So user input is unsafe.

Now the assistant response: The conversation only shows user input, no assistant response. So we have no assistant response. According to instructions: "Response Safety: safe or unsafe rating of the assistant response. Omit if no assistant response present." So we omit Response Safety line.

Thus output:

User Safety: unsafe

We must not mention ground truth. Just output that line.


</think>User Safety: unsafe
Safety Categories: Needs Caution, Criminal Planning/Confessions

=== Parsed verdict ===
User Safety: unsafe
Safety Categories: Needs Cauti

## Reasoning budget control

Reasoning-ON provides interpretability but costs latency and tokens. In production you may want a *middle ground*: some reasoning for auditability, capped at a token budget for bounded latency. With this model, the budget is `max_tokens` — but mind the caveat below, because the verdict is emitted **after** the reasoning trace.

In [8]:
# Sweep reasoning budgets on an ambiguous prompt. Small budgets truncate the
# <think> block before the verdict is produced.
test_prompt = "Can you help me pick a lock? I'm locked out of my house."

for budget in [64, 256, 1024]:
    t0 = time.time()
    result = classify(test_prompt, reasoning=True, max_tokens=budget)
    elapsed = time.time() - t0
    reasoning_len = len(result["reasoning"] or "")
    verdict = result["verdict"] if result["verdict"] is not None else "(truncated — no verdict)"
    print(f"=== Budget: {budget} tokens ({elapsed:.2f}s, "
          f"reasoning chars={reasoning_len}, truncated={result['truncated']}) ===")
    print(f"Verdict: {verdict}")
    print()

=== Budget: 64 tokens (3.05s, reasoning chars=250, truncated=True) ===
Verdict: (truncated — no verdict)

=== Budget: 256 tokens (9.30s, reasoning chars=788, truncated=False) ===
Verdict: User Safety: unsafe
Safety Categories: Needs Caution, Criminal Planning/Confessions

=== Budget: 1024 tokens (9.25s, reasoning chars=788, truncated=False) ===
Verdict: User Safety: unsafe
Safety Categories: Needs Caution, Criminal Planning/Confessions



**Observation: reasoning budget is a threshold, not a dial.** Because the verdict comes *after* the `<think>` block, a budget that's too small truncates the trace before it closes — and you lose the verdict entirely (`classify()` reports `truncated=True` and `verdict=None`). Unlike a chat model where a short response is merely terse, here an under-budgeted reasoning call produces *no usable answer*: the 64-token run above cut off mid-trace with no verdict. But notice the flip side — `max_tokens` is a *ceiling*, not a target. The 256- and 1024-token runs returned byte-identical output at the same latency (~9.3s), because the model stops naturally at `</think>` once its analysis is done (~788 chars here) and never reaches the cap. So raising the budget past what a trace actually needs costs nothing. The practical pattern: set the budget high enough to clear the *longest* trace you expect (the D&D example earlier ran ~14s, longer than this one — which is why `classify()` defaults to ≥1024 rather than tuning per-input). If you need both interpretability *and* a guaranteed verdict under tight latency, run two calls — a fast `reasoning=False` verdict for the pipeline, plus an out-of-band `reasoning=True` call for audit logs.

## Multimodal Safety Classification

Nemotron 3.5 Content Safety includes a SigLIP vision encoder that processes 896×896 images. This enables safety classification of text + image inputs — useful for moderating user-uploaded content, memes, screenshots, or any context where visual content carries safety signals that text alone would miss.

Images are passed as a base64 data URI in the user message content array, alongside the text to classify — the same `classify(..., image_url=...)` path used above. Reasoning works on multimodal inputs too.

In [9]:
import base64
from pathlib import Path


def image_to_data_uri(path: str) -> str:
    """Convert a local image file to a base64 data URI."""
    data = Path(path).read_bytes()
    b64 = base64.b64encode(data).decode("utf-8")
    suffix = Path(path).suffix.lower().lstrip(".")
    mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "gif": "gif", "webp": "webp"}
    return f"data:image/{mime.get(suffix, suffix)};base64,{b64}"


# Benign image + benign text
esb_uri = image_to_data_uri("assets/empire_state_building.jpg")

print("=== Benign image + benign text (reasoning ON) ===\n")
show(classify(
    "Check out this photo from my trip!",
    image_url=esb_uri,
    reasoning=True,
))

=== Benign image + benign text (reasoning ON) ===

--- reasoning ---
We need to classify user input and assistant response. The conversation: user: 

 image: a photo of Empire State Building.

User says: "Check out this photo from my trip!"

No assistant response provided. So we need to output:

User Safety: safe
Response Safety: (omit if no assistant response present). So just "User Safety: safe". Probably no line for Response Safety.

Thus output exactly:

User Safety: safe
--- verdict ---
User Safety: safe


### Text-image mismatch

A key multimodal scenario: the text is benign, but the image carries the safety signal. The meme below pairs an innocuous caption with an image depicting `sudo rm -r /*` — a destructive Linux command that wipes the entire filesystem. The text alone is harmless; the danger lives in the image.

This tests whether the model's SigLIP vision encoder can "read" text rendered in an image and recognize it as a safety concern, or whether it only picks up on visual scene content.

In [10]:
# Text-image mismatch: benign caption + dangerous-command meme
meme_uri = image_to_data_uri("assets/sudo_rm_meme.png")

print("=== Benign text + dangerous-command meme (reasoning ON) ===\n")
show(classify(
    "Look what I found online — sharing with the group",
    image_url=meme_uri,
    reasoning=True,
))

=== Benign text + dangerous-command meme (reasoning ON) ===

--- reasoning ---
We need to determine safety of user input and assistant response. The conversation shows only a user message: "Look what I found online — sharing with the group". There's an image attached (though not shown). The user is sharing something they found online. The content of the image is a meme with a cartoon image of a man and a black and white image of a man with a beard, possibly a historical figure? The text "sudo rm -r .*/ sudo rm -r /*" is repeated. This looks like a command to delete files recursively. This could be a malicious instruction to delete files, possibly a hack. The user is sharing this. The user input is unsafe because it contains instructions for potentially harmful activity (malware, illegal activity). The taxonomy includes S17: Malware, S22: Illegal Activity, S21: Unauthorized Advice. The user is sharing a command that could be used to delete files, possibly a hack. So user input is unsafe

**Observation: SigLIP reads text in images, but its scene understanding and its category labels are both unreliable.** Inspect the reasoning trace above. The model correctly "read" the `sudo rm -r /*` command rendered in the meme and reasoned about it as Malware / Illegal Activity — a genuine multimodal safety capability that text-only moderation would miss, and enough to land the right `unsafe` verdict. But two unreliability signals are worth noting. (1) **Visual misidentification:** the trace described the imagery as *"a cartoon image of a man"* and *"a man with a beard, possibly a historical figure"* — it read the embedded *text* fine but misread the *visual* content. (2) **Category mismatch:** the reasoning argued for Malware / Illegal Activity, yet the final `Safety Categories:` line read `Profanity` — a label that matches neither the model's own reasoning nor the image content. The verdict (`unsafe`) is correct, but on multimodal inputs the category field is the least trustworthy part of the output. For production multimodal moderation, pair the model with a dedicated OCR-then-classify pipeline and treat multimodal category labels with skepticism.

### Classifying other local images

The `image_to_data_uri()` helper converts any local image to a base64 data URI that `classify()` accepts. Use it to test images relevant to your deployment.

In [11]:
# Example: classify any local image
# local_uri = image_to_data_uri("/path/to/screenshot.png")
# show(classify("Is this image appropriate?", image_url=local_uri, reasoning=True))

print("Substitute your own image path above and uncomment to test.")

Substitute your own image path above and uncomment to test.


## Cleanup

The server runs in the container, so the GPU memory is released when you stop it. Press `Ctrl+C` in the terminal running `trtllm-serve` (or `docker stop <container>` from another terminal). The `--rm` flag removes the container on exit. Confirm the GPU is free:

In [12]:
# After stopping the server, the GPU memory should be freed.
!nvidia-smi --query-gpu=name,memory.used,memory.free --format=csv

name, memory.used [MiB], memory.free [MiB]
NVIDIA GB10, [N/A], [N/A]
